# Airbnb Vienna — Regression Analysis
## BAN Regression Project | Group 3
---

## Executive Summary

This project applies the SEMMA methodology (Sample, Explore, Modify, Model, Assess) to a dataset of Airbnb listings from Vienna, Austria, sourced from [Inside Airbnb](https://insideairbnb.com/get-the-data/). The objective is to build and progressively improve a linear regression model that predicts **nightly listing price** (in EUR).

Vienna's Airbnb market is a compelling case study: a tourist-heavy city where pricing is influenced by room type, host activity, and listing availability. The dataset contains 14,123 listings across 23 neighbourhoods.

**Key Findings:**
- Outlier removal was the single most impactful improvement, reducing MSE from ~74,958 to ~1,830.
- Room type (Private vs. Entire home) is the strongest categorical predictor of price.
- Our best model (Model 2: baseline after outlier removal) achieves **R² = 0.1315**, indicating that 13.15% of price variance is explained by the available numeric and categorical features.
- The model's limited explanatory power reflects a fundamental challenge: price is partly set by hosts subjectively, and key drivers such as amenities, photos, and proximity to landmarks are absent from the dataset.

**Business Recommendation:** Hosts seeking to price competitively should prioritise listing as an "Entire home/apt", maintain high availability, and reduce minimum night requirements to maximise review velocity — all factors the model identifies as statistically significant.


---
# 1. SAMPLE

## 1.1 Data Source

The dataset was downloaded from [Inside Airbnb](https://insideairbnb.com/get-the-data/) — specifically the Vienna, Austria listings CSV. It contains 14,123 rows and 18 columns covering listing attributes, host information, location coordinates, room type, pricing, and review statistics.

The target variable for this regression is **`price`** (nightly EUR cost). We selected price because it is a continuous numeric variable with clear business relevance — understanding what drives Airbnb pricing is valuable for both hosts (revenue optimisation) and guests (value assessment).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")

In [ ]:
# Load the dataset
df = pd.read_csv('listings.csv')

print(f"Dataset shape: {df.shape}")
df.head(10)

---
# 2. EXPLORE

## 2.1 Initial Data Inspection

Before any modelling, we examine the structure of the dataset: dimensions, data types, and missing values. This informs every subsequent decision about column treatment.

In [ ]:
# Shape of the dataset
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

In [ ]:
# Data types for each column
df.dtypes

In [ ]:
# Missing value audit — critical before any modelling
df.isna().sum()

## 2.2 Column-Level Documentation

Below is a full treatment decision for every column in the raw dataset:

| Column | Type | Missing | Decision | Rationale |
|---|---|---|---|---|
| `id` | int | 0 | **Dropped** | Arbitrary unique identifier; no predictive value |
| `name` | str | 0 | **Dropped** | Free text; Vienna names lack rating/bedroom patterns found in LA dataset |
| `host_id` | int | 0 | **Dropped** | Arbitrary host identifier; would require groupby aggregation to be meaningful |
| `host_name` | str | 3 | **Dropped** | Not relevant to pricing; personal data |
| `neighbourhood_group` | float | 14,123 | **Dropped** | 100% missing — entirely uninformative |
| `neighbourhood` | str | 0 | **Dropped** | 23 categories; dummy encoding would risk multicollinearity with room_type. Lat/long capture location implicitly |
| `latitude` | float | 0 | **Dropped** | Continuous coordinate; non-linear spatial effects require a different model type (e.g., spatial regression) |
| `longitude` | float | 0 | **Dropped** | Same reasoning as latitude |
| `room_type` | str | 0 | **Kept + Dummy Encoded** | Strong business logic: entire homes command higher prices than private or shared rooms |
| `price` | int | 3,817 | **Target variable** | Rows with missing price dropped — cannot impute a target variable |
| `minimum_nights` | int | 0 | **Kept** | Affects booking frequency and guest profile; potentially correlated with price |
| `number_of_reviews` | int | 0 | **Kept initially; later dropped (p>0.05)** | Proxy for popularity; but collinear with reviews_per_month |
| `last_review` | str | 2,289 | **Used to filter rows, then dropped** | Listings with no last_review date are inactive/unlisted; not comparable. Column itself is a date, cannot be imputed meaningfully |
| `reviews_per_month` | float | 2,289 | **Kept** | Filled automatically after row-level filter; captures listing momentum |
| `calculated_host_listings_count` | int | 0 | **Kept** | Distinguishes professional hosts (many listings) from hobbyists; relevant to pricing strategy |
| `availability_365` | int | 0 | **Kept** | Measures how actively the listing is being offered; relates to host pricing strategy |
| `number_of_reviews_ltm` | int | 0 | **Kept** | Recent review activity (last twelve months); more temporally relevant than total count |
| `license` | float | 14,123 | **Dropped** | 100% missing — entirely uninformative |


---
# 3. MODIFY

## 3.1 Column Removal

In [ ]:
# Drop columns that are 100% missing (no information value)
if 'neighbourhood_group' in df.columns:
    df.drop(columns=['neighbourhood_group'], inplace=True)
if 'license' in df.columns:
    df.drop(columns=['license'], inplace=True)

print(f"Shape after dropping empty columns: {df.shape}")
df.isna().sum()

## 3.2 Row Filtering — Removing Inactive Listings

`last_review` is a date column that is `NaN` when a listing has never been reviewed. These listings are likely inactive, unlisted, or brand new — they are not representative of the active Vienna Airbnb market. We cannot impute a date, so we drop these rows.

After this step, `reviews_per_month` NaNs are also resolved (they shared the same rows).

In [ ]:
# Drop rows where last_review is missing (inactive listings)
df.dropna(subset=['last_review'], inplace=True)

print(f"Shape after removing inactive listings: {df.shape}")
print(f"Remaining missing values:\n{df.isna().sum()}")

In [ ]:
# Drop rows where price is missing (cannot impute a target variable)
df.dropna(subset=['price'], inplace=True)

print(f"Final shape after price cleanup: {df.shape}")

## 3.3 Feature Selection & Dummy Encoding

We drop identifiers and columns unsuitable for linear regression, then dummy-encode `room_type`. The base (dropped) category is "Entire home/apt" — coefficients for the other room types represent the price premium/discount relative to renting an entire home.

In [ ]:
# Drop identifiers and columns that won't be modelled
cols_to_drop = ['id', 'name', 'host_id', 'host_name', 'neighbourhood',
                'latitude', 'longitude', 'last_review']
df.drop(columns=cols_to_drop, inplace=True)

# Dummy encode room_type (drop_first avoids perfect multicollinearity)
df = pd.get_dummies(df, columns=['room_type'], drop_first=True)

print(f"Shape after feature selection: {df.shape}")
print("\nColumns remaining:")
print(df.columns.tolist())
df.head()

---
# 4. VISUALISATION & ANALYSIS

We build a minimum of 10 visualisations, spanning univariate, bivariate, and multivariate analysis. Each is interpreted in a business context.

## Visualisation 1 — Price Distribution (Histogram)

Understanding the distribution of our target variable is the first step. A right-skewed price distribution is typical in accommodation marketplaces: most listings cluster at affordable prices, with a long tail of premium properties.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['price'], kde=True, bins=60, color='steelblue')
plt.title('Distribution of Nightly Price (EUR) — Vienna Airbnb', fontsize=14)
plt.xlabel('Price (EUR)')
plt.ylabel('Frequency')
plt.axvline(df['price'].mean(), color='red', linestyle='--', label=f"Mean: €{df['price'].mean():.0f}")
plt.axvline(df['price'].median(), color='green', linestyle='--', label=f"Median: €{df['price'].median():.0f}")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean: €{df['price'].mean():.2f} | Median: €{df['price'].median():.2f} | Std Dev: €{df['price'].std():.2f}")
print(f"Min: €{df['price'].min()} | Max: €{df['price'].max()}")

**Interpretation:** The price distribution is strongly right-skewed, with a mean (€157) significantly higher than the median (€93). This indicates a small number of very high-priced listings pull the mean upward. The long right tail (up to €10,000) represents ultra-premium or incorrectly entered listings. This skewness motivates outlier removal before modelling.

## Visualisation 2 — Price by Room Type (Boxplot)

Room type is the most intuitively meaningful categorical predictor. Entire homes should command higher prices than private or shared rooms.

In [ ]:
# Reconstruct room type labels from dummies for visualisation
df_viz = df.copy()
df_viz['Room Type'] = 'Entire home/apt'
if 'room_type_Hotel room' in df.columns:
    df_viz.loc[df['room_type_Hotel room'] == True, 'Room Type'] = 'Hotel room'
if 'room_type_Private room' in df.columns:
    df_viz.loc[df['room_type_Private room'] == True, 'Room Type'] = 'Private room'
if 'room_type_Shared room' in df.columns:
    df_viz.loc[df['room_type_Shared room'] == True, 'Room Type'] = 'Shared room'

plt.figure(figsize=(10, 5))
order = ['Entire home/apt', 'Hotel room', 'Private room', 'Shared room']
sns.boxplot(x='Room Type', y='price', data=df_viz[df_viz['price'] <= 500],
            order=order, palette='Set2')
plt.title('Price Distribution by Room Type (capped at €500 for clarity)', fontsize=13)
plt.xlabel('Room Type')
plt.ylabel('Nightly Price (EUR)')
plt.tight_layout()
plt.show()

print(df_viz.groupby('Room Type')['price'].median().sort_values(ascending=False))

**Interpretation:** Entire homes and hotel rooms command significantly higher median prices than private or shared rooms. The wide spread in 'Entire home/apt' indicates high variance — from budget studios to luxury apartments. This confirms that `room_type` should be dummy-encoded and included in our model.

## Visualisation 3 — Availability Distribution (Histogram)

Availability_365 measures how many days per year a listing is open for bookings. A bimodal distribution would suggest two distinct host strategies.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['availability_365'], bins=50, kde=True, color='coral')
plt.title('Distribution of Annual Availability (Days)', fontsize=13)
plt.xlabel('Availability (days per year)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

print(df['availability_365'].describe())

**Interpretation:** The bimodal distribution reveals two distinct host strategies: listings that are almost never available (likely personal properties offered occasionally) and listings that are nearly always available (professional hosts). This dichotomy has pricing implications — always-available listings may price more competitively to fill calendars.

## Visualisation 4 — Minimum Nights Distribution (Histogram)

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df[df['minimum_nights'] <= 30]['minimum_nights'], bins=30, kde=False, color='mediumpurple')
plt.title('Minimum Nights Requirement Distribution (≤30 nights shown)', fontsize=13)
plt.xlabel('Minimum Nights')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

print(df['minimum_nights'].describe())
print(f"\n>30 nights: {(df['minimum_nights'] > 30).sum()} listings ({(df['minimum_nights'] > 30).mean()*100:.1f}%)")

**Interpretation:** The vast majority of listings require only 1–3 nights minimum, consistent with a tourist accommodation market. A small tail of long-term minimum stay listings (30+ nights) represents medium-term rentals — a different product category that may introduce noise into our price model.

## Visualisation 5 — Reviews Per Month Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df[df['reviews_per_month'] <= 10]['reviews_per_month'], bins=40, kde=True, color='seagreen')
plt.title('Reviews Per Month Distribution (≤10 shown)', fontsize=13)
plt.xlabel('Reviews Per Month')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

print(df['reviews_per_month'].describe())

**Interpretation:** Reviews per month is heavily right-skewed — most listings receive less than one review per month, while a small number of high-turnover listings receive several monthly. Since reviews are typically left after checkout, this variable is a reasonable proxy for booking frequency.

## Visualisation 6 — Correlation Matrix (Heatmap)

A correlation matrix identifies linear relationships between all numeric variables, flagging potential multicollinearity issues before modelling.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Matrix — Numeric Variables', fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation:**
- **`number_of_reviews` and `number_of_reviews_ltm`** show moderate correlation (expected: recent reviews are a subset of total reviews). This is worth monitoring for multicollinearity.
- **`reviews_per_month` and `number_of_reviews`** are moderately correlated (0.40+), as highly reviewed listings naturally accumulate more total reviews.
- **`price`** shows weak correlations with all other variables — a hallmark of real-world pricing data where many unobserved factors (amenities, photos, décor) drive price.
- **No extreme collinearity** (|r| > 0.8) was detected between independent variables, so we can proceed with the full feature set.


## Visualisation 7 — Price vs. Availability (Scatterplot)

In [ ]:
plt.figure(figsize=(10, 5))
sample = df[df['price'] <= 500].sample(2000, random_state=42)
sns.scatterplot(x='availability_365', y='price', data=sample, alpha=0.3, color='steelblue')
sns.regplot(x='availability_365', y='price', data=sample, scatter=False, color='red', line_kws={'linewidth': 2})
plt.title('Price vs. Annual Availability', fontsize=13)
plt.xlabel('Availability (days/year)')
plt.ylabel('Price (EUR)')
plt.tight_layout()
plt.show()

print(f"Correlation: {df['availability_365'].corr(df['price']):.4f}")

**Interpretation:** The regression line reveals a slight negative relationship: listings with higher annual availability tend to be priced lower. This is consistent with professional hosts pricing competitively to maintain high occupancy. Purely occasional hosts may price premium for their limited slots.

## Visualisation 8 — Price vs. Reviews Per Month (Scatterplot)

In [ ]:
plt.figure(figsize=(10, 5))
sample2 = df[(df['price'] <= 500) & (df['reviews_per_month'] <= 10)].sample(2000, random_state=42)
sns.scatterplot(x='reviews_per_month', y='price', data=sample2, alpha=0.3, color='coral')
sns.regplot(x='reviews_per_month', y='price', data=sample2, scatter=False, color='darkred', line_kws={'linewidth': 2})
plt.title('Price vs. Reviews Per Month', fontsize=13)
plt.xlabel('Reviews Per Month')
plt.ylabel('Price (EUR)')
plt.tight_layout()
plt.show()

print(f"Correlation: {df['reviews_per_month'].corr(df['price']):.4f}")

**Interpretation:** A slight negative relationship exists — higher-frequency review listings (i.e., more bookings) tend to be moderately priced. Very high-priced listings receive fewer reviews per month, either because they are booked less frequently or because guests at premium properties may review less consistently.

## Visualisation 9 — Listing Count by Room Type (Bar Chart)

In [ ]:
room_counts = df_viz['Room Type'].value_counts()

plt.figure(figsize=(9, 5))
bars = plt.bar(room_counts.index, room_counts.values,
               color=['steelblue', 'coral', 'seagreen', 'mediumpurple'])
plt.title('Number of Active Listings by Room Type', fontsize=13)
plt.xlabel('Room Type')
plt.ylabel('Number of Listings')
for bar, val in zip(bars, room_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{val:,}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

print(room_counts)
print(f"\nEntire homes represent {room_counts['Entire home/apt']/room_counts.sum()*100:.1f}% of active listings")

**Interpretation:** The overwhelming majority of active Vienna listings are entire homes (~82%), followed by private rooms (~17%). Shared rooms and hotel rooms are negligible in count. This market composition means price modelling is primarily driven by entire-home pricing dynamics, with private rooms forming a structurally different sub-market.

## Visualisation 10 — Host Listings Count Distribution (Pie Chart)

Understanding whether the market is dominated by professional multi-listers or casual hosts matters for pricing strategy.

In [ ]:
host_bins = pd.cut(df['calculated_host_listings_count'],
                   bins=[0, 1, 5, 20, 100, 10000],
                   labels=['1 listing', '2–5 listings', '6–20 listings',
                           '21–100 listings', '100+ listings'])
host_bin_counts = host_bins.value_counts().sort_index()

plt.figure(figsize=(8, 8))
plt.pie(host_bin_counts, labels=host_bin_counts.index,
        autopct='%1.1f%%', colors=sns.color_palette('Set2', 5),
        startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
plt.title('Distribution of Hosts by Number of Listings', fontsize=13)
plt.tight_layout()
plt.show()

print(host_bin_counts)

**Interpretation:** A substantial share of listings come from hosts managing multiple properties, suggesting a professional rental operator segment. Professional hosts typically price more strategically (dynamic pricing, competitive rates) compared to one-off hosts. The model's `calculated_host_listings_count` feature captures this distinction.

## Visualisation 11 — Boxplot: Price by Minimum Nights Bucket

In [ ]:
df_viz2 = df_viz[df_viz['price'] <= 500].copy()
df_viz2['min_nights_bucket'] = pd.cut(df_viz2['minimum_nights'],
    bins=[0, 1, 3, 7, 30, 9999],
    labels=['1 night', '2–3 nights', '4–7 nights', '8–30 nights', '30+ nights'])

plt.figure(figsize=(11, 5))
sns.boxplot(x='min_nights_bucket', y='price', data=df_viz2, palette='Set3')
plt.title('Price Distribution by Minimum Nights Requirement', fontsize=13)
plt.xlabel('Minimum Nights Category')
plt.ylabel('Price (EUR)')
plt.tight_layout()
plt.show()

print(df_viz2.groupby('min_nights_bucket')['price'].median())

**Interpretation:** Listings requiring 30+ nights have notably different (often lower) nightly prices — they are medium-term rentals, not tourist short-stays. Including these without treatment adds noise to the model. The broad 1-night to 7-night range shows relatively consistent median prices, validating that short-term listings form a coherent pricing group.

---
# 5. MODEL BUILDING

## 5.1 Feature Preparation for Modelling

We define our feature set. The dummy-encoded room type columns are included. `number_of_reviews` is retained at this stage for the baseline model, though p-value analysis will guide further refinement.

In [ ]:
features = [
    'minimum_nights',
    'number_of_reviews',
    'reviews_per_month',
    'calculated_host_listings_count',
    'availability_365',
    'number_of_reviews_ltm',
    'room_type_Hotel room',
    'room_type_Private room',
    'room_type_Shared room'
]

X = df[features]
y = df['price']

print(f"Feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nTarget variable stats:")
print(y.describe())

---
## MODEL 1 — Baseline (Full Dataset, No Pre-processing)

Our first model uses the full cleaned dataset with all features and a 70/30 train-test split. This establishes a performance baseline we can measure all improvements against.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model1 = LinearRegression()
model1.fit(X_train, y_train)

y_pred1 = model1.predict(X_test)
mse1 = mean_squared_error(y_test, y_pred1)
r2_1 = r2_score(y_test, y_pred1)
mor1 = (y_test - y_pred1).mean()

print(f"=== MODEL 1: Baseline ===")
print(f"R²:             {r2_1:.4f}")
print(f"MSE:            {mse1:,.2f}")
print(f"Mean Residual:  {mor1:.4f}")
print(f"\nCoefficients:")
for feat, coef in zip(features, model1.coef_):
    print(f"  {feat:45s}: {coef:+.4f}")
print(f"  Intercept: {model1.intercept_:.4f}")

**Model 1 Interpretation:**
The baseline model performs very poorly with an R² of ~0.010 — it explains less than 1% of price variance. The MSE of ~74,958 is enormous, driven by extreme outliers in the price distribution (some listings priced at €10,000/night). The mean residual of -7.2 indicates the model systematically under-predicts. This is expected: we have not treated outliers yet.


---
## MODEL 2 — Outlier Removal (Primary Model)

The single biggest issue with Model 1 is extreme price outliers. We apply the IQR method to remove price outliers, which creates a cleaner, more representative dataset for linear regression.

**Why prioritise outlier removal?** Linear regression is sensitive to outliers — a single listing priced at €10,000 exerts disproportionate leverage on the coefficient estimates, pulling predictions away from the bulk of the data.

In [ ]:
# IQR-based outlier removal on the target variable (price)
Q1 = y.quantile(0.25)
Q3 = y.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"IQR: {IQR:.2f} | Lower bound: €{lower_bound:.2f} | Upper bound: €{upper_bound:.2f}")

mask = (y >= lower_bound) & (y <= upper_bound)
df_filtered = df[mask].copy()
X_filtered = df_filtered[features]
y_filtered = df_filtered['price']

removed = len(df) - len(df_filtered)
print(f"Listings removed as outliers: {removed:,} ({removed/len(df)*100:.1f}%)")
print(f"Remaining dataset size: {len(df_filtered):,}")
print(f"\nPrice statistics after outlier removal:")
print(y_filtered.describe())

In [ ]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_filtered, y_filtered, test_size=0.3, random_state=42)

model2 = LinearRegression()
model2.fit(X_train2, y_train2)
y_pred2 = model2.predict(X_test2)

mse2 = mean_squared_error(y_test2, y_pred2)
r2_2 = r2_score(y_test2, y_pred2)
mor2 = (y_test2 - y_pred2).mean()

print(f"=== MODEL 2: Outlier Removal ===")
print(f"R²:             {r2_2:.4f}")
print(f"MSE:            {mse2:,.2f}")
print(f"Mean Residual:  {mor2:.6f}")
print(f"\nCoefficients:")
for feat, coef in zip(features, model2.coef_):
    print(f"  {feat:45s}: {coef:+.4f}")
print(f"  Intercept: {model2.intercept_:.4f}")

In [ ]:
# Residual distribution plot
residuals2 = y_test2 - y_pred2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(residuals2, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', label='Zero residual')
axes[0].set_title('Model 2 — Residual Distribution')
axes[0].set_xlabel('Residual (Actual − Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].scatter(y_pred2, residuals2, alpha=0.3, color='coral')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Model 2 — Residuals vs. Fitted Values')
axes[1].set_xlabel('Fitted (Predicted) Price')
axes[1].set_ylabel('Residual')
plt.tight_layout()
plt.show()

print(f"Residual mean: {residuals2.mean():.6f}")
print(f"Residual std:  {residuals2.std():.4f}")

**Model 2 Interpretation:**
Outlier removal dramatically improves all metrics:
- **R² = 0.1315**: The model now explains 13.15% of price variance — a 13× improvement over the baseline.
- **MSE = 1,830**: Down from ~74,958 — a 97.6% reduction.
- **Mean Residual ≈ 0.04**: Effectively zero, confirming the model has no systematic directional bias.

The residuals are approximately centred at zero but remain spread out, indicating unexplained variance. This is expected: price is partly subjective and many important features (amenities, photos, host responsiveness) are absent from the dataset.


---
## MODEL 3 — Standardisation

Standardisation (Z-score scaling) transforms features to have mean=0 and std=1. This does not change the predictive power of OLS linear regression in terms of R² or MSE, but it enables **coefficient comparison** — all coefficients become expressed in terms of standard deviations, making their relative importance directly comparable.

**Why use it?** Our features have very different scales: `minimum_nights` ranges from 1–1,125 while `room_type_Private room` is binary (0 or 1). Without standardisation, comparing raw coefficients is misleading.

In [ ]:
scaler = StandardScaler()
X_train3 = scaler.fit_transform(X_train2)
X_test3 = scaler.transform(X_test2)

model3 = LinearRegression()
model3.fit(X_train3, y_train2)
y_pred3 = model3.predict(X_test3)

mse3 = mean_squared_error(y_test2, y_pred3)
r2_3 = r2_score(y_test2, y_pred3)
mor3 = (y_test2 - y_pred3).mean()

print(f"=== MODEL 3: Standardisation (on outlier-removed data) ===")
print(f"R²:             {r2_3:.4f}")
print(f"MSE:            {mse3:,.2f}")
print(f"Mean Residual:  {mor3:.6f}")
print(f"\nStandardised Coefficients (magnitude = relative importance):")
for feat, coef in sorted(zip(features, model3.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:45s}: {coef:+.4f}")

**Model 3 Interpretation:**
As expected for OLS, R² and MSE are identical to Model 2 — standardisation does not change predictive performance. However, the standardised coefficients reveal **relative feature importance**:
- `room_type_Private room` has the largest magnitude coefficient, confirming room type is the dominant driver.
- `reviews_per_month` and `minimum_nights` are also meaningful negative predictors.
- `number_of_reviews` has a very small coefficient, suggesting it adds little beyond what `number_of_reviews_ltm` already captures.


---
## MODEL 4 — P-Value Significance Filtering + Standardisation

Not all features are statistically significant. We use OLS from statsmodels to compute p-values, then retain only features with p < 0.05 (the conventional 5% significance threshold). This creates a more parsimonious model.

In [ ]:
# Compute p-values using statsmodels OLS
X_sm = sm.add_constant(X_train2.astype(float))
ols_model = sm.OLS(y_train2.astype(float), X_sm).fit()

print(ols_model.summary())

In [ ]:
# Extract significant features
p_values = ols_model.pvalues.drop('const')
print("P-values for all features:")
print(p_values.to_string())

significant_features = p_values[p_values < 0.05].index.tolist()
insignificant_features = p_values[p_values >= 0.05].index.tolist()

print(f"\n✓ Significant (p < 0.05): {significant_features}")
print(f"✗ Insignificant (p ≥ 0.05): {insignificant_features}")

In [ ]:
# Rebuild model with only significant features
X_sig = df_filtered[significant_features]
y_sig = df_filtered['price']

X_train4, X_test4, y_train4, y_test4 = train_test_split(X_sig, y_sig, test_size=0.3, random_state=42)

scaler4 = StandardScaler()
X_train4s = scaler4.fit_transform(X_train4)
X_test4s = scaler4.transform(X_test4)

model4 = LinearRegression()
model4.fit(X_train4s, y_train4)
y_pred4 = model4.predict(X_test4s)

mse4 = mean_squared_error(y_test4, y_pred4)
r2_4 = r2_score(y_test4, y_pred4)
mor4 = (y_test4 - y_pred4).mean()

print(f"=== MODEL 4: Significant Features + Standardisation ===")
print(f"R²:             {r2_4:.4f}")
print(f"MSE:            {mse4:,.2f}")
print(f"Mean Residual:  {mor4:.6f}")
print(f"\nFeatures used ({len(significant_features)}): {significant_features}")

**Model 4 Interpretation:**
Dropping `number_of_reviews` and `room_type_Hotel room` (both p > 0.05) produces a marginally lower R² (~0.131). The trade-off is worth it: a simpler, more interpretable model that only includes variables with demonstrated statistical significance. In a business setting, a parsimonious model is preferred for actionability — every variable left in must earn its place.

**Why were `number_of_reviews` and `room_type_Hotel room` insignificant?**
- `number_of_reviews` is largely redundant with `number_of_reviews_ltm` — recent activity is more informative.
- `room_type_Hotel room` represents only ~40 listings in the dataset — too few for the model to reliably estimate its effect.


---
## MODEL 5 — Significant Features + Standardisation + Reduced Test Size (20%)

Our final model tests whether giving the model more training data (80% instead of 70%) improves performance. More training data can improve coefficient stability, especially for smaller categories like `Shared room`.

In [ ]:
X_train5, X_test5, y_train5, y_test5 = train_test_split(
    X_sig, y_sig, test_size=0.2, random_state=42)  # 80/20 split

scaler5 = StandardScaler()
X_train5s = scaler5.fit_transform(X_train5)
X_test5s = scaler5.transform(X_test5)

model5 = LinearRegression()
model5.fit(X_train5s, y_train5)
y_pred5 = model5.predict(X_test5s)

mse5 = mean_squared_error(y_test5, y_pred5)
r2_5 = r2_score(y_test5, y_pred5)
mor5 = (y_test5 - y_pred5).mean()

print(f"=== MODEL 5: Sig. Features + Std + 80/20 Split ===")
print(f"R²:             {r2_5:.4f}")
print(f"MSE:            {mse5:,.2f}")
print(f"Mean Residual:  {mor5:.6f}")
print(f"\nTraining set size: {len(X_train5):,} | Test set size: {len(X_test5):,}")

---
# 6. MODEL ASSESSMENT

## 6.1 Comparison Across All Models

In [ ]:
results = pd.DataFrame({
    'Model': ['Model 1 (Baseline)', 'Model 2 (Outliers Removed)',
              'Model 3 (Standardised)', 'Model 4 (Sig. Features + Std)',
              'Model 5 (80/20 + Std)'],
    'R²':   [r2_1, r2_2, r2_3, r2_4, r2_5],
    'MSE':  [mse1, mse2, mse3, mse4, mse5],
    'Mean Residual': [mor1, mor2, mor3, mor4, mor5],
    'Technique': ['Baseline', 'Outlier Removal', 'Standardisation',
                  'P-value Filter + Std', 'Reduced Test Size']
})

print(results.to_string(index=False))

In [ ]:
# Visual comparison heatmap
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['R²', 'MSE', 'Mean Residual']
for ax, metric in zip(axes, metrics):
    vals = results[metric].values.reshape(-1, 1)
    # Normalise for colour (exclude MSE from colour scaling directly)
    sns.heatmap(pd.DataFrame(results[metric].values, index=results['Model'], columns=[metric]),
                annot=True, fmt='.4f', cmap='RdYlGn' if metric == 'R²' else 'RdYlGn_r',
                ax=ax, cbar=True, linewidths=0.5)
    ax.set_title(metric)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.suptitle('Model Performance Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6.2 Selected Model & Rationale

**We select Model 2 (Outlier Removal) as our best model.**

| Metric | Model 2 Value | Interpretation |
|---|---|---|
| R² | 0.1315 | 13.15% of price variance explained |
| MSE | 1,830 | Average squared error of ~€42.8 per prediction (RMSE = √1830) |
| Mean Residual | ~0.04 | Model has no systematic directional bias |

**Why Model 2 over Models 3–5?**
- Models 3, 4, and 5 show marginal and sometimes negative changes in R² compared to Model 2.
- Standardisation (Model 3) does not improve R² — it's a scaling transformation for interpretability only.
- P-value filtering (Model 4) slightly reduces R² as we sacrifice some variance explanation for parsimony.
- The 80/20 split (Model 5) does not meaningfully change performance, confirming the model is not data-starved.

**Which success metric are we prioritising, and why?**

We prioritise **R²** as our primary success metric because:
1. R² is scale-independent, making it comparable across datasets and models.
2. In a business context, R² answers the direct question: *"How much of price variability does our model explain?"*
3. MSE is important for operational precision (actual EUR error magnitude), but a very low MSE can be achieved by simply predicting the mean — R² rewards genuine explanatory power.
4. Mean of Residuals (MOR) is a sanity check for directional bias, not a measure of quality. A near-zero MOR is necessary but not sufficient for a good model.

**Translating for a business manager:** Our best model explains roughly 1 in 8 euros of price variation across Vienna Airbnb listings. The model is unbiased (it doesn't consistently over- or under-predict), and its average absolute error is roughly €43 per night. For listings in the €64–€140 price band (where 50% of active listings sit), this represents a meaningful but imprecise prediction — useful for benchmarking, not for setting prices to the cent.


## 6.3 P-Value & Coefficient Analysis

In [ ]:
# Full OLS summary for Model 2 (the selected model)
X_sm2 = sm.add_constant(X_train2.astype(float))
ols2 = sm.OLS(y_train2.astype(float), X_sm2).fit()
print(ols2.summary())

## 6.4 Translating Statistical Results for Business Managers

**R² (R-squared):** Think of this as a "fit score". An R² of 0.13 means our model captures 13% of the reasons why one listing costs more than another. The remaining 87% is driven by factors we don't have data on — quality of photos, amenities, host responsiveness, and proximity to major attractions.

**MSE (Mean Squared Error):** This measures our prediction error. Our MSE of 1,830 translates to an average absolute prediction error of roughly €43 (the square root). For a listing priced at €100, we might predict anywhere from €57 to €143 — useful for market positioning analysis, but not for precise pricing.

**Mean of Residuals (MOR):** With a MOR of ~0.04, our model is perfectly unbiased on average. It doesn't systematically overprice or underprice — errors in both directions cancel out.

**P-values:** A p-value tells us how confident we are that a variable actually affects price (and isn't just coincidence in our data). All variables in Model 2 with p < 0.05 are statistically significant. The two insignificant variables (`number_of_reviews` and `room_type_Hotel room`) were removed in Model 4.

**Multicollinearity:** We checked for collinearity via the correlation matrix (Visualisation 6). No feature pair exceeded |r| = 0.8 among predictors, so multicollinearity is not a material concern in this model. The moderate correlation between `number_of_reviews` and `number_of_reviews_ltm` was addressed by the p-value filter in Model 4, which dropped the total review count.


## 6.5 Price Equation (Selected Model)

In [ ]:
# Model 2 coefficients (unstandardised — raw EUR impact per unit)
print("Model 2 — Price Prediction Equation:")
print("=" * 60)
print(f"Price = {model2.intercept_:.2f}")
for feat, coef in zip(features, model2.coef_):
    print(f"      + ({coef:+.4f}) × {feat}")
print()
print("Simplified:")
print(f"  Each additional 'minimum night' required  → {model2.coef_[0]:+.2f} EUR")
print(f"  Each additional review per month          → {model2.coef_[2]:+.2f} EUR")
print(f"  Each additional host listing              → {model2.coef_[3]:+.2f} EUR")
print(f"  Each additional available day             → {model2.coef_[4]:+.2f} EUR")
print(f"  Being a 'Private room' vs Entire home     → {model2.coef_[7]:+.2f} EUR")
print(f"  Being a 'Shared room' vs Entire home      → {model2.coef_[8]:+.2f} EUR")

---
# 7. BUSINESS RECOMMENDATIONS

Based on our regression analysis of 8,302 active Vienna Airbnb listings, we offer the following actionable recommendations:

## For Hosts — Revenue Optimisation

**1. List as an Entire Home whenever possible (+€27 premium vs. Private Room)**
The model's strongest predictor is room type. Entire home listings command a significant premium over private and shared rooms. Hosts who can restructure their offering (e.g., by vacating during peak season) stand to gain meaningfully.

**2. Reduce minimum night requirements**
The negative coefficient on `minimum_nights` confirms that high minimum stays are associated with lower prices — likely because hosts targeting long-term guests must discount to compete with the apartment rental market. Short-stay pricing should target the 1–3 night sweet spot.

**3. Maximise availability to attract competitive pricing signals**
Higher availability (availability_365) is positively associated with price, though modestly. Professional hosts who keep their calendar open attract higher-frequency guests, supporting market-rate pricing.

**4. Focus on recent review velocity, not total review count**
`number_of_reviews_ltm` was statistically significant; total `number_of_reviews` was not. Recency of social proof matters more than historical volume. Hosts should actively encourage recent guests to leave reviews.

## For Airbnb / Platform Strategy

**5. The model's limited R² (13%) signals a data gap**
Price is substantially explained by factors absent from the public dataset: amenity quality, interior photos, instant booking, and superhosts status. Airbnb should consider providing richer anonymised data for academic research while investing in enhanced search ranking algorithms that factor in these hidden quality signals.

**6. Vienna's bimodal availability pattern suggests two market segments**
The bimodal availability distribution (Visualisation 3) reveals two distinct host types: occasional hosts and professional operators. Platform features (pricing tools, dynamic pricing recommendations) should be differentiated for these two segments.

## Model Limitations & Improvement Suggestions

- **Non-linear models** (Random Forest, Gradient Boosting) would likely capture the complex, non-linear interactions between price drivers far better than OLS.
- **Feature engineering**: Adding distance-to-landmarks (e.g., Stephansdom, Schönbrunn Palace), seasonality variables, or text sentiment from listing names could substantially boost explanatory power.
- **Log transformation of price**: Given the right-skewed distribution, modelling log(price) rather than raw price typically improves OLS fit.
- **Spatial features**: Neighbourhood coordinates could be used to compute spatial lag variables — capturing the effect of being in a high-demand district.
